# Interest Rates and Yield Curves

Interest rates are the price of time and credit. Yield curves organize rates by maturity and are foundational for discounting, bond pricing, valuation, and macro-financial analysis.

Abbreviations used in this notebook:

- **YTM**: Yield to Maturity.
- **PV**: Present Value.
- **DF**: Discount Factor.
- **bps**: Basis Points, where 100 bps equals 1 percentage point.

## 1. Intuition

A yield curve shows the interest rate investors require for lending money over different maturities. The x-axis is time to maturity, such as 1 year, 2 years, 5 years, or 10 years. The y-axis is the yield for that maturity.

Short rates are influenced strongly by central bank policy. Long rates reflect expectations about future short rates, inflation, term premia, and risk appetite. Longer maturities often have higher yields because investors give up their money for longer, face more inflation uncertainty, and hold bonds that are more sensitive to interest-rate changes.

Common yield curve shapes:

- **Normal upward-sloping curve**: long-term yields are higher than short-term yields. This is common when investors expect normal growth and inflation.
- **Flat curve**: short-term and long-term yields are similar. This can indicate uncertainty or a transition period.
- **Inverted curve**: short-term yields are higher than long-term yields. This can happen when central banks raise short-term rates while investors expect future rates, inflation, or growth to fall.

Yield curves matter because they provide maturity-specific discount rates. A cash flow received in 1 year should be discounted with a 1-year rate. A cash flow received in 10 years should be discounted with a 10-year rate. Bond pricing, DCF valuation, and credit spread analysis all build on this idea.

Key rate concepts:

- **Short rate**: interest rate for a very short lending period.
- **Spot rate**: zero-coupon rate for a specific maturity.
- **Forward rate**: rate implied today for a future period.
- **Discount factor**: present value of one unit of currency received in the future.
- **Yield curve slope**: difference between long-term and short-term rates.
- **Credit spread**: extra yield over a reference curve for credit risk.


## 2. Mathematics

**Discount factor from a spot rate:**

$$
DF_t = \frac{1}{(1 + s_t)^t}
$$

Where:

- $DF_t$ = discount factor for maturity $t$
- $s_t$ = annual spot rate for maturity $t$
- $t$ = maturity in years

**Present value using a discount factor:**

$$
PV = CF_t \times DF_t
$$

Where:

- $PV$ = present value today
- $CF_t$ = cash flow received at maturity $t$
- $DF_t$ = discount factor for maturity $t$

**One-year forward rate from spot rates:**

$$
1 + f_{1,2} = \frac{(1 + s_2)^2}{(1 + s_1)}
$$

Where:

- $f_{1,2}$ = one-year forward rate starting in one year
- $s_1$ = one-year spot rate
- $s_2$ = two-year spot rate

**Yield curve slope:**

$$
\text{Slope} = y_{10Y} - y_{2Y}
$$

Where:

- $y_{10Y}$ = yield for a 10-year maturity
- $y_{2Y}$ = yield for a 2-year maturity
- $\text{Slope}$ = difference between long-term and short-term yields


## 3. Implementation

We will build a small synthetic yield curve, calculate discount factors, derive one-year forward rates, and show the curve slope.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.4f}".format

In [ ]:
curve = pd.DataFrame({
    "maturity_years": [1, 2, 3, 5, 7, 10],
    "spot_rate": [0.018, 0.021, 0.023, 0.026, 0.028, 0.031],
})

curve["discount_factor"] = 1 / (1 + curve["spot_rate"]) ** curve["maturity_years"]
curve["pv_of_chf_100"] = 100 * curve["discount_factor"]

curve

In [ ]:
one_year_points = curve[curve["maturity_years"].isin([1, 2, 3])].copy()
forward_1y_1y = ((1 + one_year_points.loc[one_year_points["maturity_years"] == 2, "spot_rate"].iloc[0]) ** 2 / (1 + one_year_points.loc[one_year_points["maturity_years"] == 1, "spot_rate"].iloc[0])) - 1
forward_2y_1y = ((1 + one_year_points.loc[one_year_points["maturity_years"] == 3, "spot_rate"].iloc[0]) ** 3 / (1 + one_year_points.loc[one_year_points["maturity_years"] == 2, "spot_rate"].iloc[0]) ** 2) - 1

slope_10y_2y = curve.loc[curve["maturity_years"] == 10, "spot_rate"].iloc[0] - curve.loc[curve["maturity_years"] == 2, "spot_rate"].iloc[0]

rate_summary = pd.DataFrame({
    "metric": [
        "1Y spot rate",
        "2Y spot rate",
        "10Y spot rate",
        "1Y forward rate starting in 1Y",
        "1Y forward rate starting in 2Y",
        "10Y minus 2Y slope",
    ],
    "value": [
        f"{curve.loc[curve['maturity_years'] == 1, 'spot_rate'].iloc[0]:.2%}",
        f"{curve.loc[curve['maturity_years'] == 2, 'spot_rate'].iloc[0]:.2%}",
        f"{curve.loc[curve['maturity_years'] == 10, 'spot_rate'].iloc[0]:.2%}",
        f"{forward_1y_1y:.2%}",
        f"{forward_2y_1y:.2%}",
        f"{slope_10y_2y * 10_000:.0f} bps",
    ],
})

rate_summary

## 4. Visualization

Yield curves are usually displayed as maturity on the x-axis and yield on the y-axis.

The examples below compare common curve shapes before plotting the synthetic spot curve used for discounting.


In [ ]:
curve_shapes = pd.DataFrame({
    "maturity_years": [1, 2, 5, 10],
    "normal": [0.020, 0.023, 0.028, 0.032],
    "flat": [0.030, 0.030, 0.031, 0.031],
    "inverted": [0.045, 0.042, 0.036, 0.032],
})

fig, ax = plt.subplots(figsize=(9, 4))
for column, color in [("normal", "#2f6f8f"), ("flat", "#6f8f4e"), ("inverted", "#9a3d2f")]:
    ax.plot(curve_shapes["maturity_years"], curve_shapes[column], marker="o", label=column.title(), color=color)

ax.set_title("Common Yield Curve Shapes")
ax.set_xlabel("Maturity in years")
ax.set_ylabel("Yield")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(curve["maturity_years"], curve["spot_rate"], marker="o", color="#2f6f8f")
axes[0].set_title("Synthetic Spot Rate Curve")
axes[0].set_xlabel("Maturity in years")
axes[0].set_ylabel("Spot rate")
axes[0].yaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")

axes[1].plot(curve["maturity_years"], curve["discount_factor"], marker="o", color="#9a6b2f")
axes[1].set_title("Discount Factors")
axes[1].set_xlabel("Maturity in years")
axes[1].set_ylabel("Present value of CHF 1")

plt.tight_layout()
plt.show()

## 5. Application

Yield curves are used to discount cash flows, price bonds, interpret market expectations, compare credit spreads, and stress-test valuation models under different rate scenarios.

### How Yield Curves Reach Risk Systems

Yield curves used by banks usually start from raw market quotes such as deposits, overnight indexed swaps, swap rates, futures, government bonds, credit instruments, and broker or trading-platform quotes. Market data vendors and trading venues provide many of these inputs, but banks often construct and validate their own official curves internally.

Typical flow:

$$
\text{Raw Market Quotes} \rightarrow \text{Market Data Platform} \rightarrow \text{Curve Construction Engine} \rightarrow \text{Validated Curves} \rightarrow \text{Pricing and Risk Systems}
$$

Where:

- $\text{Raw Market Quotes}$ = observed rates, prices, and spreads from vendors, venues, brokers, or internal trading systems
- $\text{Market Data Platform}$ = system that stores, normalizes, and validates market data inputs
- $\text{Curve Construction Engine}$ = model or library that bootstraps/interpolates curves from market inputs
- $\text{Validated Curves}$ = approved curves used consistently across valuation, risk, accounting, and limits
- $\text{Pricing and Risk Systems}$ = systems that consume curves for valuation, sensitivities, VaR, stress testing, and reporting

Vendors may provide ready-made curves, but banks often define their own curve methodology so pricing and risk use consistent assumptions. Detailed bootstrapping, interpolation, and multi-curve construction belong in a quantitative methods notebook.


## 6. Reflection

- A yield curve organizes interest rates by maturity.
- Discount factors convert future cash flows into present values.
- Forward rates are implied by spot rates.
- Curve slope is often used as a macro and market signal.
- Production yield curves usually come from raw market inputs that are constructed, validated, and published into pricing and risk systems.

Questions to answer after running the notebook:

1. Why does a higher discount rate reduce present value?
2. What information does a forward rate contain?
3. Why might long-term yields differ from short-term yields?
4. How do yield curves connect to bond pricing and DCF valuation?